# Africa’s Fertility Paradox (2000–2024)

This notebook follows the original analysis: **MySQL** prepares and exports data, while **Python** calculates Pearson correlations and creates visualizations. 

**Scope:** 54 African countries. **Data:** World Bank indicators, supplied as original project SQL-result CSV exports. 

## 0. Setup and file paths

All inputs are in `data/analysis_outputs/`. Run the notebook from the project root or `python/` folder.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Open the notebook from the project root, or from the python/ directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "python":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "analysis_outputs"
CHART_DIR = ROOT / "visualizations"
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Cannot find the exports folder: {DATA_DIR}")
CHART_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
NAVY = "#346C91"
ORANGE = "#EA785F"


def load_csv(filename, required_columns):
    """Read an existing MySQL export; preserve its original filename."""
    data = pd.read_csv(DATA_DIR / filename, encoding="utf-8-sig")
    missing = set(required_columns) - set(data.columns)
    if missing:
        raise ValueError(f"{filename}: missing columns {sorted(missing)}")
    return data


def finish_chart(fig, filename, note):
    """Save a chart in the same folder shown in the README."""
    fig.text(0.10, 0.015, note, fontsize=8, color="#555555")
    fig.tight_layout(rect=(0, 0.055, 1, 1))
    fig.savefig(CHART_DIR / filename, dpi=180, bbox_inches="tight")
    plt.show()


## 1. Mean fertility trends (2000–2024)

Unweighted country averages are *not* population-weighted rates for Africa as a whole.

In [ ]:
mean_data = load_csv("annual_mean_fertility_2000_2024.csv", ["year", "avg_fertility_rate"])
mean_data = mean_data.sort_values("year")
start, end = mean_data.iloc[0], mean_data.iloc[-1]
change_pct = 100 * (end.avg_fertility_rate / start.avg_fertility_rate - 1)

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.lineplot(data=mean_data, x="year", y="avg_fertility_rate", marker="o",
             markersize=4.5, linewidth=2.4, color=NAVY, ax=ax)
ax.scatter([start.year, end.year], [start.avg_fertility_rate, end.avg_fertility_rate],
           color=ORANGE, s=90, zorder=5)
ax.annotate(f"{start.avg_fertility_rate:.2f}", (start.year, start.avg_fertility_rate),
            xytext=(7, 10), textcoords="offset points", fontweight="bold")
ax.annotate(f"{end.avg_fertility_rate:.2f}", (end.year, end.avg_fertility_rate),
            xytext=(-36, 12), textcoords="offset points", fontweight="bold")
ax.text(0.98, 0.90, f"Change: {change_pct:.1f}%", transform=ax.transAxes,
        ha="right", fontsize=12, fontweight="bold")
ax.set(title="Average fertility across 54 African countries (2000–2024)",
       xlabel="Year", ylabel="Unweighted mean (births per woman)")
finish_chart(fig, "annual_mean_fertility_2000_2024_polished.png",
             "Source: World Bank data, exported from project MySQL analysis. Means are unweighted across countries.")
print(f"2000: {start.avg_fertility_rate:.3f}; 2024: {end.avg_fertility_rate:.3f}; change: {change_pct:.1f}%")


## 2. GDP per capita and fertility (2024)

Pearson correlation is computed **in Python** using pandas. 

In [ ]:
gdp_2024 = load_csv("gpd_vs_fertility_2024.csv", ["gdp_per_capita", "fertility_rate"])
gdp_2024 = gdp_2024.dropna(subset=["gdp_per_capita", "fertility_rate"])
r = gdp_2024["gdp_per_capita"].corr(gdp_2024["fertility_rate"])

fig, ax = plt.subplots(figsize=(9.5, 5.6))
sns.regplot(data=gdp_2024, x="gdp_per_capita", y="fertility_rate", ax=ax,
            color=NAVY, scatter_kws={"s": 48, "alpha": 0.85}, ci=None,
            line_kws={"color": ORANGE, "linewidth": 2})
ax.set(title="GDP per capita and fertility (2024)", xlabel="GDP per capita (constant 2015 US$)",
       ylabel="Fertility rate (births per woman)")
ax.text(0.97, 0.92, f"n = {len(gdp_2024)}  ·  Pearson r = {r:.3f}",
        transform=ax.transAxes, ha="right", fontweight="bold")
finish_chart(fig, "gdp_vs_fertility_2024_polished.png",
             "Source: World Bank, World Development Indicators (WDI)")
print(f"GDP–fertility, 2024: n={len(gdp_2024)}, Pearson r={r:.3f}")


### GDP–fertility comparison: 2000 and 2024

This visualization compares the relationship between GDP per capita and fertility across African countries in 2000 and 2024. It does not examine changes within individual countries or establish a causal relationship.

In [ ]:
gdp_comparison = load_csv("gpd_vs_fertility_2000_2024.csv", ["year", "gdp_per_capita", "fertility_rate"])
fig, ax = plt.subplots(figsize=(9.5, 5.6))
for year, color, marker in [(2000, NAVY, "o"), (2024, ORANGE, "X")]:
    subset = gdp_comparison[gdp_comparison["year"] == year]
    subset = subset.dropna(subset=["gdp_per_capita", "fertility_rate"])
    r = subset["gdp_per_capita"].corr(subset["fertility_rate"])
    ax.scatter(subset.gdp_per_capita, subset.fertility_rate, color=color,
               marker=marker, s=40, alpha=0.8, label=f"{year}: n={len(subset)}, r={r:.3f}")
    print(f"GDP–fertility, {year}: n={len(subset)}, Pearson r={r:.3f}")
ax.set(title="GDP per capita and fertility: 2000 vs. 2024",
       xlabel="GDP per capita (constant 2015 US$)", ylabel="Fertility rate (births per woman)")
ax.legend(title="Year and Pearson correlation", loc="upper right")
finish_chart(fig, "gdp_vs_fertility_2000_2024_polished.png",
             "Source: World Bank, World Development Indicators (WDI)")


## 3. Female secondary enrollment and fertility (2011)

The year **2011** was selected because female secondary school enrollment data were available for 37 African countries, providing better coverage than in 2024. The indicator measures the gross enrollment ratio, which includes female students of all ages relative to the female population of official secondary school age. Consequently, enrollment ratios can exceed 100%.

In [ ]:
education = load_csv("gpd_vs_edu_2011.csv", ["female_secondary_enrollment", "fertility_rate"])
education = education.dropna(subset=["female_secondary_enrollment", "fertility_rate"])
r= education["female_secondary_enrollment"].corr(education["fertility_rate"])

fig, ax = plt.subplots(figsize=(9.5, 5.6))
sns.regplot(data=education, x="female_secondary_enrollment", y="fertility_rate", ax=ax,
            color=NAVY, scatter_kws={"s": 52, "alpha": 0.85}, ci=None,
            line_kws={"color": ORANGE, "linewidth": 2})
ax.set(title="Female secondary enrollment and fertility (2011)",
       xlabel="Female secondary gross enrollment (%)", ylabel="Births per woman")
ax.text(0.97, 0.92, f"n = {len(education)}  ·  Pearson r = {r:.3f}",
        transform=ax.transAxes, ha="right", fontweight="bold")
finish_chart(fig, "female_education_vs_fertility_2011_polished.png",
             "Source: World Bank, World Development Indicators (WDI)")
print(f"Education–fertility, 2011: n={len(education)}, Pearson r={r:.3f}")


## 4. Urbanization and fertility (2024)

Correlation is calculated using pandas on the SQL-exported sample.

In [ ]:
urban_2024 = load_csv("gpd_vs_urbanization_2024.csv", ["urban_pop_pct", "fertility_rate"])
urban_2024 = urban_2024.dropna(subset=["urban_pop_pct", "fertility_rate"])
r = urban_2024["urban_pop_pct"].corr(urban_2024["fertility_rate"])

fig, ax = plt.subplots(figsize=(9.5, 5.6))
sns.regplot(data=urban_2024, x="urban_pop_pct", y="fertility_rate", ax=ax,
            color=NAVY, scatter_kws={"s": 52, "alpha": 0.85}, ci=None,
            line_kws={"color": ORANGE, "linewidth": 2})
ax.set(title="Urban population share and fertility (2024)",
       xlabel="Urban population (% of total)", ylabel="Births per woman")
ax.text(0.97, 0.92, f"n = {len(urban_2024)}  ·  Pearson r = {r:.3f}",
        transform=ax.transAxes, ha="right", fontweight="bold")
finish_chart(fig, "urbanization_vs_fertility_2024_polished.png",
             "Source: World Bank, World Development Indicators (WDI)")
print(f"Urbanization–fertility, 2024: n={len(urban_2024)}, Pearson r={r:.3f}")


### Urbanization–fertility comparison: 2000 and 2024

In [ ]:
urban_comparison = load_csv("gpd_vs_urbanization_2000_2024.csv", ["year", "urban_pop_pct", "fertility_rate"])
fig, ax = plt.subplots(figsize=(9.5, 5.6))
for year, color, marker in [(2000, NAVY, "o"), (2024, ORANGE, "X")]:
    subset = urban_comparison.loc[urban_comparison.year == year].dropna(
        subset=["urban_pop_pct", "fertility_rate"])
    r = subset["urban_pop_pct"].corr(subset["fertility_rate"])
    ax.scatter(subset.urban_pop_pct, subset.fertility_rate, color=color,
               marker=marker, s=40, alpha=0.8, label=f"{year}: n={len(subset)}, r={r:.3f}")
    print(f"Urbanization–fertility, {year}: n={len(subset)}, Pearson r={r:.3f}")
ax.set(title="Urban population share and fertility: 2000 vs. 2024",
       xlabel="Urban population (% of total)", ylabel="Births per woman")
ax.legend(title="Year and Pearson correlation", loc="upper right")
finish_chart(fig, "urbanization_vs_fertility_2000_2024_polished.png",
             "Source: World Bank, World Development Indicators (WDI).")


## 5. Fertility outliers among GDP peers (2024)

MySQL computed z-scores within **three GDP peer groups** using the **population standard deviation**. The plot reads the original six-row export; it does not recalculate peer membership.

In [ ]:
outliers = load_csv("africa_fertility_outliers_2024.csv", ["country", "fertility_zscore", "deviation_direction"])
outliers = outliers.sort_values("fertility_zscore")
colors = [NAVY if z < 0 else ORANGE for z in outliers.fertility_zscore]
fig, ax = plt.subplots(figsize=(9.5, 5.5))
ax.barh(outliers.country, outliers.fertility_zscore, color=colors, height=0.68)
ax.axvline(0, color="#333333", linewidth=1.2)
ax.axvline(-1.5, color="#888888", linestyle="--", linewidth=1)
ax.axvline(1.5, color="#888888", linestyle="--", linewidth=1)
for i, value in enumerate(outliers.fertility_zscore):
    ax.text(value + (0.08 if value > 0 else -0.08), i, f"{value:+.2f}",
            ha="left" if value > 0 else "right", va="center", fontweight="bold")
ax.set_xlim(-2.65, 2.9)
ax.set(title="Fertility outliers among GDP-based peers (2024)",
       xlabel="Within-group fertility z-score", ylabel="")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=NAVY, label="Lower than peers"),
                   Patch(facecolor=ORANGE, label="Higher than peers")],
          loc="upper left")
finish_chart(fig, "outliers_2024_polished.png",
             "Source: 2024 MySQL export; 52 countries, NTILE(3) GDP groups. Dashed lines: |z| = 1.5.")
print(f"Displayed {len(outliers)} of 52 eligible 2024 countries; the sensitivity check uses four groups.")


## Interpretation and limitations

- Average fertility fell from **5.191** (2000) to **3.795** (2024), a **26.9%** decline in the unweighted cross-country mean.
- The 2024 GDP and urbanization correlations are **−0.647** (n=52) and **−0.345** (n=54), respectively.
- The female secondary gross enrollment correlation is **−0.903 in 2011** (n=37), not in 2024.
- Outlier identification depends on GDP groupings. Only **Angola and Lesotho** occur in both the three- and four-group lists.
- Missing data change samples; GDP source units should be checked against the original World Bank indicator metadata.
- These results are **associations, not causal effects**. The project does not test whether outliers persist across years.

See `docs/validation_record.md` for checks and `data/README.md` for original CSV provenance.